# Spotify Audio Feature-Based Music Genre Prediction

This notebook presents an end-to-end machine learning pipeline for predicting Spotify music genres from audio features.

## Project summary
- **Task:** multi-class music genre classification
- **Dataset:** `musicData.csv`
- **Random seed:** `13000000`
- **Evaluation:** One-vs-Rest ROC curves and macro-averaged AUC
- **Final model:** multinomial logistic regression

## Main findings
According to the capstone report, the final multinomial logistic regression model achieved a **macro OvR AUC of 0.898**, outperforming the calibrated linear SVM (**0.881**). The report also notes that the most important performance gain came from **clean preprocessing, no-leakage handling, and PCA on numeric audio features**.

## Notebook structure
1. Load and clean the data  
2. Build a leakage-safe preprocessing pipeline  
3. Run PCA and analyze explained variance  
4. Add KMeans cluster features  
5. Train and compare logistic regression vs. calibrated linear SVM  
6. Evaluate with OvR ROC/AUC and error analysis  

> Note: This notebook is a cleaned, portfolio-ready version of the project notebook, organized for readability on GitHub.


## 0. Setup
Import libraries, set the random seed, and define the project-wide generator.


In [ ]:
import random
import numpy as np
N_NUMBER_SEED = 13000000
random.seed(N_NUMBER_SEED)
np.random.seed(N_NUMBER_SEED)
rng_exp = np.random.default_rng(N_NUMBER_SEED)


## 1. Load the dataset and inspect basic structure
This step loads the dataset, fixes invalid values in `duration_ms` and `tempo`, and prints a first-pass inspection of shape, data types, and missingness.


In [ ]:
# =========================================================
# Step 1: Load data and document feature types
# =========================================================
import pandas as pd

# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------
df = pd.read_csv("musicData.csv")

df.loc[df["duration_ms"] <= 0, "duration_ms"] = np.nan
df["tempo"] = pd.to_numeric(df["tempo"], errors="coerce")


print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isna().sum())

# ---------------------------------------------------------
# 2. Explicitly document feature types
# ---------------------------------------------------------

# Identifier / metadata columns (not used for modeling)
identifier_cols = [
    "instance_id",
    "artist_name",
    "track_name",
    "obtained_date"
]

df["tempo"] = pd.to_numeric(df["tempo"], errors="coerce")


# Numeric audio features (continuous)
numeric_cols = [
    "popularity",
    "acousticness",
    "danceability",
    "duration_ms",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence"
]

# Categorical features (non-numeric meaning)
categorical_cols = [
    "key",
    "mode"
]

# Target variable
target_col = "music_genre"

# ---------------------------------------------------------
# 3. Feature-type summary
# ---------------------------------------------------------
feature_summary = {
    "Identifiers (drop later)": identifier_cols,
    "Numeric features (scale later)": numeric_cols,
    "Categorical features (encode later)": categorical_cols,
    "Target": [target_col]
}

print("\nFeature type summary:")
for k, v in feature_summary.items():
    print(f"{k}:")
    for col in v:
        print(f"  - {col}")
    print()

# ---------------------------------------------------------
# 4. Sanity checks
# ---------------------------------------------------------
all_listed_cols = (
    identifier_cols +
    numeric_cols +
    categorical_cols +
    [target_col]
)

unclassified_cols = set(df.columns) - set(all_listed_cols)

print("Unclassified columns (should be empty):")
print(unclassified_cols)


## 2. Remove non-predictive identifier columns
These columns are dropped because they are metadata or identifiers rather than stable audio signals, and they may encourage memorization or leakage.


In [ ]:
# =========================================================
# Step 2: Drop non-predictive identifier / metadata columns
# =========================================================

# Identifier / metadata columns to remove
identifier_cols = [
    "instance_id",
    "artist_name",
    "track_name",
    "obtained_date"
]

print("Dropping non-predictive identifier columns:")
for col in identifier_cols:
    print(f"  - {col}")

# Drop identifiers
df_model = df.drop(columns=identifier_cols)

print("\nShape before dropping identifiers:", df.shape)
print("Shape after dropping identifiers:", df_model.shape)

print("\nRemaining columns:")
print(df_model.columns.tolist())


## 3. Define features and target
Prepare numeric, categorical, and target columns, then remove rows with missing target labels.


In [ ]:
# =========================================================
# Step 3: Prepare data (no-leakage safe)
# =========================================================

numeric_cols = [
    "popularity","acousticness","danceability","duration_ms","energy",
    "instrumentalness","liveness","loudness","speechiness","valence"
]
categorical_cols = ["key","mode"]
tempo_col = "tempo"
target_col = "music_genre"

# Drop rows with missing target
df_model = df_model.dropna(subset=[target_col])

# Type conversion only (safe pre-split)
numeric_cols = numeric_cols + [tempo_col]

# Diagnostic only
print("Missing values per column (pre-split, before imputation):")
print(df_model.isna().sum())


## 4. One-hot encode categorical variables
Convert categorical columns into numeric features so they can be used by PCA-compatible and linear classification models.


In [ ]:
# =========================================================
# Step 4: Encode categorical variables (no scaling)
# =========================================================
import pandas as pd
# Columns (based on your dataset)
categorical_cols = ["key", "mode"]
target_col = "music_genre"
# Numeric columns are everything else except the target
numeric_cols = [c for c in df_model.columns if c not in categorical_cols + [target_col]]
print("Numeric columns (no scaling in Step 4):")
print(numeric_cols)
print("\nCategorical columns to one-hot encode:")
print(categorical_cols)
# ---------------------------------------------------------
# 1. Build X (features) and y (target)
# ---------------------------------------------------------
X_raw = df_model[numeric_cols + categorical_cols].copy()
y_raw = df_model[target_col].copy()
# ---------------------------------------------------------
# 2. One-hot encode categorical variables (no scaling)
# ---------------------------------------------------------
X_encoded = pd.get_dummies(
    X_raw,
    columns=categorical_cols,
    drop_first=False,   # keep all categories for interpretability
    dummy_na=False      # missing values will be imputed later (TRAIN-only) after the split
)
print("\nShape before encoding:", X_raw.shape)
print("Shape after encoding :", X_encoded.shape)
# Sanity checks
print("\nMissing values per column in X_encoded (top 10):")
print(X_encoded.isna().sum().sort_values(ascending=False).head(10))
print("Target class count:", y_raw.nunique())
# Optional: preview encoded column names
print("\nExample encoded columns (first 30):")
print(X_encoded.columns[:30].tolist())


## 5. Create a leakage-safe train/test split
This split follows the project specification and uses a fixed seed for reproducibility.


In [ ]:
# =========================================================
# Step 5: Spec-compliant train/test split (500 test per genre)
# =========================================================
import random
import numpy as np
import pandas as pd

# -----------------------------
# 0) Set seed using YOUR N-number (do this ONCE and do not overwrite)
# -----------------------------
N_NUMBER_SEED = 13000000

# Spec-safe: seed Python + NumPy
random.seed(N_NUMBER_SEED)
np.random.seed(N_NUMBER_SEED)

# Modern NumPy RNG for any randomness in the experiment
rng_exp = np.random.default_rng(N_NUMBER_SEED)

# -----------------------------
# 1) Sanity checks
# -----------------------------
y = y_raw.copy()
X = X_encoded.copy()

genre_counts = y.value_counts()
print("Genre counts (full dataset):")
print(genre_counts)

# Ensure each genre has at least 500 samples for the test set requirement
if (genre_counts < 500).any():
    raise ValueError(
        "At least one genre has fewer than 500 samples; cannot perform the required split.\n"
        f"Problem genres:\n{genre_counts[genre_counts < 500]}"
    )

# -----------------------------
# 2) Sample exactly 500 test rows per genre (deterministic)
# -----------------------------
test_idx_list = []

for genre, idx in y.groupby(y).groups.items():
    idx = np.array(list(idx))
    chosen = rng_exp.choice(idx, size=500, replace=False)
    test_idx_list.append(chosen)

test_idx = pd.Index(np.concatenate(test_idx_list))

# Train indices are all remaining rows
train_idx = X.index.difference(test_idx)

# -----------------------------
# 3) Build train/test splits
# -----------------------------
X_train = X.loc[train_idx].copy()
X_test  = X.loc[test_idx].copy()

y_train = y.loc[train_idx].copy()
y_test  = y.loc[test_idx].copy()

# -----------------------------
# 4) Verify split is spec-compliant
# -----------------------------
print("\nSplit sizes:")
print("  X_train:", X_train.shape, " y_train:", y_train.shape)
print("  X_test :", X_test.shape,  " y_test :", y_test.shape)

print("\nGenre counts in TEST (must be exactly 500 each):")
print(y_test.value_counts().sort_index())

assert (y_test.value_counts() == 500).all(), "TEST set does not have exactly 500 per genre."

print("\nGenre counts in TRAIN (should be remaining per genre):")
print(y_train.value_counts().sort_index())

# -----------------------------
# 5) Optional: Shuffle row order (membership unchanged)
# -----------------------------
train_perm = rng_exp.permutation(len(X_train))
X_train = X_train.iloc[train_perm].reset_index(drop=True)
y_train = y_train.iloc[train_perm].reset_index(drop=True)

test_perm = rng_exp.permutation(len(X_test))
X_test = X_test.iloc[test_perm].reset_index(drop=True)
y_test = y_test.iloc[test_perm].reset_index(drop=True)

print("\nDone: spec-compliant split created with no leakage.")




print("NaNs in X_train before scaling:", X_train.isna().sum().sum())
print("NaNs in X_test before scaling :", X_test.isna().sum().sum())


## 6. Impute missing numeric values using train-set medians only
Median imputation is fit on the training set and then applied to both training and test data.


In [ ]:
# =========================================================
# Step 5.5: Impute missing numeric values (fit on TRAIN only)
# =========================================================

numeric_cols = [
    "popularity",
    "acousticness",
    "danceability",
    "duration_ms",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence"
]

# Fit imputer on TRAIN only
numeric_medians = X_train[numeric_cols].median()

# Apply to TRAIN
X_train[numeric_cols] = X_train[numeric_cols].fillna(numeric_medians)

# Apply SAME values to TEST
X_test[numeric_cols] = X_test[numeric_cols].fillna(numeric_medians)

# Verify
print("NaNs after imputation TRAIN:", X_train.isna().sum().sum())
print("NaNs after imputation TEST :", X_test.isna().sum().sum())


## 7. Standardize numeric features only
Numeric features are scaled for PCA and linear models. One-hot encoded categorical features are left unscaled.


In [ ]:
# =========================================================
# Step 6 (REVISED): Standardize numeric features only (train-only fit)
#   - Do NOT scale one-hot categorical features
#   - Prepare separate matrices for PCA (numeric-only) and later concat
# =========================================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# -----------------------------
# 0) Define columns
# -----------------------------
numeric_cols = [
    "popularity","acousticness","danceability","duration_ms","energy",
    "instrumentalness","liveness","loudness","speechiness","tempo","valence"
]

# one-hot categorical columns = everything else (already dummy-coded in Step 4)
onehot_cols = [c for c in X_train.columns if c not in numeric_cols]

# -----------------------------
# 1) Sanity checks
# -----------------------------
missing_num_cols_train = [c for c in numeric_cols if c not in X_train.columns]
missing_num_cols_test  = [c for c in numeric_cols if c not in X_test.columns]
if missing_num_cols_train or missing_num_cols_test:
    raise ValueError(
        "Numeric columns missing from X_train/X_test.\n"
        f"Missing in train: {missing_num_cols_train}\n"
        f"Missing in test : {missing_num_cols_test}"
    )

# Ensure onehot columns match between train/test
missing_onehot_in_test = [c for c in onehot_cols if c not in X_test.columns]
if missing_onehot_in_test:
    raise ValueError(
        "Some one-hot columns exist in TRAIN but not in TEST.\n"
        f"Missing in test: {missing_onehot_in_test[:20]} (showing first 20)"
    )

print("Numeric cols to scale:", numeric_cols)
print("One-hot cols NOT scaled (count):", len(onehot_cols))

# -----------------------------
# 2) Fit scaler on TRAIN numeric only; transform TRAIN/TEST
# -----------------------------
scaler = StandardScaler()

X_train_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index
)

X_test_num_scaled = pd.DataFrame(
    scaler.transform(X_test[numeric_cols]),
    columns=numeric_cols,
    index=X_test.index
)

# -----------------------------
# 3) Keep one-hot as-is (no scaling)
# -----------------------------
X_train_onehot = X_train[onehot_cols].copy()
X_test_onehot  = X_test[onehot_cols].copy()

# -----------------------------
# 4) Quick validation (numeric only)
# -----------------------------
train_means = X_train_num_scaled.mean().round(6)
train_stds  = X_train_num_scaled.std(ddof=0).round(6)

print("\nTrain scaled numeric means (should be ~0):")
print(train_means)

print("\nTrain scaled numeric stds (should be ~1):")
print(train_stds)

print("\nShapes:")
print("X_train_num_scaled:", X_train_num_scaled.shape)
print("X_test_num_scaled :", X_test_num_scaled.shape)
print("X_train_onehot    :", X_train_onehot.shape)
print("X_test_onehot     :", X_test_onehot.shape)


## 8. PCA for dimensionality reduction
Fit PCA on the scaled numeric training data and select the number of components using a cumulative explained variance threshold.


In [ ]:
# Step 7: PCA with variance-based component selection
# =========================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

VAR_THRESHOLD = 0.90

# Fit full PCA on TRAIN numeric only to choose k
pca_full = PCA(random_state=N_NUMBER_SEED)
pca_full.fit(X_train_num_scaled)

cum_explained = np.cumsum(pca_full.explained_variance_ratio_)
k = int(np.searchsorted(cum_explained, VAR_THRESHOLD) + 1)

print(f"Selected k={k} for VAR_THRESHOLD={VAR_THRESHOLD}, cum_var={cum_explained[k-1]:.4f}")

# Plot cumulative explained variance (optional but good for report)
plt.figure()
plt.plot(range(1, len(cum_explained)+1), cum_explained, marker="o")
plt.axhline(VAR_THRESHOLD, linestyle="--")
plt.axvline(k, linestyle="--")
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA on numeric-only: cumulative explained variance (train only)")
plt.grid(True)
plt.show()

# Fit PCA(k) on TRAIN numeric only; transform TRAIN/TEST numeric only
pca = PCA(n_components=k, random_state=N_NUMBER_SEED)
X_train_pca = pca.fit_transform(X_train_num_scaled)
X_test_pca  = pca.transform(X_test_num_scaled)

pca_cols = [f"PC{i+1}" for i in range(k)]
X_train_pca = pd.DataFrame(X_train_pca, columns=pca_cols)
X_test_pca  = pd.DataFrame(X_test_pca,  columns=pca_cols)

print("PCA shapes:", X_train_pca.shape, X_test_pca.shape)


## 9. Build the base classifier feature matrix
Concatenate PCA components with one-hot categorical features.


In [ ]:
#Step 7.5 : Concatenate PCA + one-hot (this becomes your classifier input base)
# Combine PCA components + original one-hot columns
X_train_base = pd.concat([X_train_pca.reset_index(drop=True), X_train_onehot], axis=1)
X_test_base  = pd.concat([X_test_pca.reset_index(drop=True),  X_test_onehot], axis=1)

print("Base feature matrix shapes:", X_train_base.shape, X_test_base.shape)


## 10. PCA sensitivity analysis
This analysis compares multiple explained-variance thresholds to understand how dimensionality reduction affects predictive performance.


In [ ]:
# =========================================================
# Step 7B (REVISED): PCA sensitivity analysis (numeric-only)
#   - Analysis ONLY
#   - All randomness uses N_NUMBER_SEED
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# -----------------------------
# 0) Choose thresholds to compare
# -----------------------------
THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]

# -----------------------------
# 1) Encode labels (train-only fit)
# -----------------------------
le_tmp = LabelEncoder()
y_train_enc_tmp = le_tmp.fit_transform(y_train)
y_test_enc_tmp  = le_tmp.transform(y_test)
n_classes = len(le_tmp.classes_)

# -----------------------------
# 2) Fit "full PCA" ONCE on TRAIN numeric-only
# -----------------------------
pca_full_tmp = PCA(random_state=N_NUMBER_SEED)
pca_full_tmp.fit(X_train_num_scaled)

cum_explained = np.cumsum(pca_full_tmp.explained_variance_ratio_)

results = []

for var_thr in THRESHOLDS:
    # ---- choose smallest k achieving var_thr ----
    k = int(np.searchsorted(cum_explained, var_thr) + 1)

    # ---- fit PCA(k) on TRAIN numeric-only ----
    pca_k = PCA(n_components=k, random_state=N_NUMBER_SEED)
    Xtr_pca = pca_k.fit_transform(X_train_num_scaled)
    Xte_pca = pca_k.transform(X_test_num_scaled)

    # ---- baseline classifier (deterministic) ----
    clf = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=2000,
        n_jobs=-1,
        random_state=N_NUMBER_SEED
    )
    clf.fit(Xtr_pca, y_train_enc_tmp)

    # ---- Predict probabilities + compute TEST AUC ----
    y_test_proba = clf.predict_proba(Xte_pca)

    auc_macro = roc_auc_score(
        y_test_enc_tmp,
        y_test_proba,
        multi_class="ovr",
        average="macro"
    )
    auc_weighted = roc_auc_score(
        y_test_enc_tmp,
        y_test_proba,
        multi_class="ovr",
        average="weighted"
    )

    results.append({
        "variance_threshold": var_thr,
        "k_components": k,
        "cum_var_at_k": float(cum_explained[k - 1]),
        "test_auc_macro": float(auc_macro),
        "test_auc_weighted": float(auc_weighted),
    })

results_df = pd.DataFrame(results).sort_values("variance_threshold")

print("PCA sensitivity (variance threshold → AUC on TEST):")
display(results_df)

# -----------------------------
# 3) Plot (threshold vs AUC)
# -----------------------------
plt.figure()
plt.plot(results_df["variance_threshold"], results_df["test_auc_macro"], marker="o", label="Test AUC (macro)")
plt.plot(results_df["variance_threshold"], results_df["test_auc_weighted"], marker="o", label="Test AUC (weighted)")
plt.xlabel("PCA variance threshold")
plt.ylabel("OvR AUC")
plt.title("PCA sensitivity (numeric-only PCA)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 11. Interpret PCA loadings
Inspect the variables contributing most strongly to the leading principal components.


In [ ]:
# =========================================================
# Step 7C (REVISED): Interpret PCA loadings (numeric-only PCA)
# =========================================================
import pandas as pd

# PCA was fit on X_train_num_scaled (numeric-only), so loadings must use those columns
loadings = pd.DataFrame(
    pca.components_.T,
    index=X_train_num_scaled.columns,   # <-- FIX: numeric-only feature names
    columns=pca_cols
)

TOP_N = 8
PCS_TO_SHOW = min(5, len(pca_cols))  # PC1..PC5

for j in range(PCS_TO_SHOW):
    pc = pca_cols[j]
    print(f"\nTop {TOP_N} absolute loadings for {pc}:")
    display(loadings[pc].abs().sort_values(ascending=False).head(TOP_N))

# Compact summary for PC1–PC3
pcs_summary = []
for pc in pca_cols[:min(3, len(pca_cols))]:
    top_feats = loadings[pc].abs().sort_values(ascending=False).head(5).index.tolist()
    pcs_summary.append({"PC": pc, "Top_5_features": ", ".join(top_feats)})

summary_df = pd.DataFrame(pcs_summary)
print("\nCompact summary (Top features by PC):")
display(summary_df)


## 12. Add KMeans cluster features
KMeans is fit in PCA space on the training set only, then cluster assignments are appended as additional classifier inputs.


In [ ]:
# =========================================================
# Step 8 (REVISED): KMeans on PCA space + add cluster as feature
#   - Fit KMeans on TRAIN PCA only
#   - Assign clusters to TEST (no refit)
#   - One-hot encode clusters and append to features
# =========================================================

import pandas as pd
from sklearn.cluster import KMeans

# -----------------------------
# 0) Settings
# -----------------------------
K_CLUSTERS = 10  # 10 genres in dataset

# -----------------------------
# 1) Fit KMeans on TRAIN PCA space only (no leakage)
# -----------------------------
kmeans = KMeans(
    n_clusters=K_CLUSTERS,
    random_state=N_NUMBER_SEED,
    n_init=10
)

train_cluster = kmeans.fit_predict(X_train_pca)
test_cluster  = kmeans.predict(X_test_pca)

print("KMeans fit complete.")
print("Train cluster labels shape:", train_cluster.shape)
print("Test cluster labels shape :", test_cluster.shape)

# -----------------------------
# 2) One-hot encode cluster assignments
# -----------------------------
train_cluster_oh = pd.get_dummies(train_cluster, prefix="cluster")
test_cluster_oh  = pd.get_dummies(test_cluster,  prefix="cluster")

# Ensure train/test have identical cluster columns
train_cluster_oh, test_cluster_oh = train_cluster_oh.align(
    test_cluster_oh, join="outer", axis=1, fill_value=0
)

print("\nCluster one-hot shapes:")
print("train_cluster_oh:", train_cluster_oh.shape)
print("test_cluster_oh :", test_cluster_oh.shape)

# -----------------------------
# 3) Append cluster features to base feature matrices
# -----------------------------
X_train_final = pd.concat(
    [X_train_base.reset_index(drop=True), train_cluster_oh.reset_index(drop=True)],
    axis=1
)

X_test_final = pd.concat(
    [X_test_base.reset_index(drop=True), test_cluster_oh.reset_index(drop=True)],
    axis=1
)

print("\nFinal feature matrix shapes (for classification):")
print("X_train_final:", X_train_final.shape)
print("X_test_final :", X_test_final.shape)

# Optional sanity check
assert list(X_train_final.columns) == list(X_test_final.columns), \
    "Train/Test feature columns do not match!"



print("X_train_final shape:", X_train_final.shape)
print("X_test_final shape :", X_test_final.shape)
print(
    "Train/Test columns identical?",
    list(X_train_final.columns) == list(X_test_final.columns)
)
print("Any NaNs train?", X_train_final.isna().sum().sum())
print("Any NaNs test ?", X_test_final.isna().sum().sum())


## 13. Train classification models
Train the final multinomial logistic regression model and the calibrated linear SVM for comparison.


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# Encode labels (fit on TRAIN only)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

# Baseline model: train on FINAL features
baseline_clf = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=3000,
    n_jobs=-1,
    random_state=N_NUMBER_SEED
)
baseline_clf.fit(X_train_final, y_train_enc)

# Stronger model: calibrated linear SVM trained on FINAL features
base_svm = LinearSVC(C=1.0, random_state=N_NUMBER_SEED)
cal_svm = CalibratedClassifierCV(base_svm, method="sigmoid", cv=3)
cal_svm.fit(X_train_final, y_train_enc)

print("Refit complete: baseline_clf and cal_svm trained on X_train_final.")


from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder

# 1) Label encoding (fit on TRAIN only)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

# 2) Train calibrated linear SVM on FINAL features
base_svm = LinearSVC(C=1.0, random_state=N_NUMBER_SEED)
cal_svm = CalibratedClassifierCV(base_svm, method="sigmoid", cv=3)

cal_svm.fit(X_train_final, y_train_enc)

print("Calibrated SVM trained on X_train_final.")


## 14. Build analysis tables for visualization
Create helper DataFrames containing PCA coordinates, genre labels, and cluster assignments.


In [ ]:
# =========================================================
# Step 8.5: Build analysis DataFrames for visualization (PCs + genre + cluster)
# =========================================================
import pandas as pd

# Use PCA coordinates
train_analysis_df = X_train_pca.copy()
test_analysis_df  = X_test_pca.copy()

# Attach labels
train_analysis_df["genre"] = y_train.reset_index(drop=True).values
test_analysis_df["genre"]  = y_test.reset_index(drop=True).values

# Attach cluster assignments
train_analysis_df["cluster"] = pd.Series(train_cluster).reset_index(drop=True).values
test_analysis_df["cluster"]  = pd.Series(test_cluster).reset_index(drop=True).values

print("train_analysis_df:", train_analysis_df.shape)
print("test_analysis_df :", test_analysis_df.shape)
display(train_analysis_df.head())


## 15. Visualize PCA space and cluster structure
These visualizations help show how genres overlap in feature space and how KMeans partitions the PCA-reduced data.


In [ ]:
# =========================================================
# Step 9: Visualize PCA space (genre + cluster) and interpret
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 0) Config (VIZ ONLY)
# -----------------------------
PLOT_SAMPLE_TRAIN = 8000   # reduce point count for readability/speed
PLOT_SAMPLE_TEST  = 5000   # test is already small
RANDOM_SEED_VIZ = N_NUMBER_SEED


def sample_df(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(df) <= n:
        return df
    rng = np.random.default_rng(seed)
    pos = rng.choice(len(df), size=n, replace=False)
    return df.iloc[pos]

train_plot_df = sample_df(train_analysis_df, PLOT_SAMPLE_TRAIN, seed=RANDOM_SEED_VIZ)
test_plot_df  = sample_df(test_analysis_df,  PLOT_SAMPLE_TEST,  seed=RANDOM_SEED_VIZ + 1)


# -----------------------------
# 1) Plot: TRAIN PC1 vs PC2 colored by GENRE
# -----------------------------
plt.figure()
for g in sorted(train_plot_df["genre"].unique()):
    sub = train_plot_df[train_plot_df["genre"] == g]
    plt.scatter(sub["PC1"], sub["PC2"], s=8, alpha=0.6, label=g)

plt.title("TRAIN: PCA (PC1 vs PC2) colored by Genre")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

# -----------------------------
# 2) Plot: TRAIN PC1 vs PC2 colored by KMeans CLUSTER
# -----------------------------
plt.figure()
for c in sorted(train_plot_df["cluster"].unique()):
    sub = train_plot_df[train_plot_df["cluster"] == c]
    plt.scatter(sub["PC1"], sub["PC2"], s=8, alpha=0.6, label=f"Cluster {c}")

plt.title("TRAIN: PCA (PC1 vs PC2) colored by KMeans Cluster")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

# -----------------------------
# 3) Plot: TEST PC1 vs PC2 colored by GENRE (optional but useful)
# -----------------------------
plt.figure()
for g in sorted(test_plot_df["genre"].unique()):
    sub = test_plot_df[test_plot_df["genre"] == g]
    plt.scatter(sub["PC1"], sub["PC2"], s=14, alpha=0.7, label=g)

plt.title("TEST: PCA (PC1 vs PC2) colored by Genre")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

# -----------------------------
# 4) Interpretation helpers
#    A) Top genres inside each cluster (train)
# -----------------------------
print("TRAIN: Top genres per cluster (counts and proportions):")
cluster_summary = []

for c in sorted(train_analysis_df["cluster"].unique()):
    sub = train_analysis_df[train_analysis_df["cluster"] == c]
    counts = sub["genre"].value_counts()
    top_genre = counts.index[0]
    top_prop = counts.iloc[0] / len(sub)
    cluster_summary.append((c, len(sub), top_genre, top_prop))

cluster_summary_df = pd.DataFrame(
    cluster_summary,
    columns=["cluster", "cluster_size", "top_genre", "top_genre_prop"]
).sort_values("top_genre_prop", ascending=False)

display(cluster_summary_df)

#    B) Genre spread across clusters (train) — helps you describe overlap
print("\nTRAIN: Genre spread across clusters (row-normalized):")
genre_cluster_train = pd.crosstab(
    train_analysis_df["genre"],
    train_analysis_df["cluster"],
    normalize="index"
)

display(genre_cluster_train)

# -----------------------------
# 5) Optional: show cluster centroids in PC1/PC2
# -----------------------------
centroids = train_analysis_df.groupby("cluster")[["PC1", "PC2"]].mean()

plt.figure()
# plot points lightly
plt.scatter(train_plot_df["PC1"], train_plot_df["PC2"], s=6, alpha=0.25)
# plot centroids
plt.scatter(centroids["PC1"], centroids["PC2"], s=180, marker="X")

for c, row in centroids.iterrows():
    plt.text(row["PC1"], row["PC2"], str(c), fontsize=10)

plt.title("TRAIN: PCA with KMeans Centroids (PC1 vs PC2)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


# =========================================================
# 4A-Visual: Cluster "purity" bar chart (Top-genre proportion per cluster)
#   - Extra-credit friendly visualization
# =========================================================

# Bar chart: how dominated each cluster is by its top genre
plt.figure(figsize=(8, 4))
plt.bar(
    cluster_summary_df["cluster"].astype(str),
    cluster_summary_df["top_genre_prop"]
)
plt.ylim(0, 1.0)
plt.xlabel("Cluster ID")
plt.ylabel("Top-genre proportion in cluster")
plt.title("TRAIN: Cluster purity (share of dominant genre per cluster)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Optional: annotate bars with the dominant genre label (keeps it readable)
plt.figure(figsize=(8, 4))
bars = plt.bar(
    cluster_summary_df["cluster"].astype(str),
    cluster_summary_df["top_genre_prop"]
)
plt.ylim(0, 1.0)
plt.xlabel("Cluster ID")
plt.ylabel("Top-genre proportion in cluster")
plt.title("TRAIN: Cluster purity + dominant genre label")
plt.grid(axis="y", alpha=0.3)

for i, row in cluster_summary_df.reset_index(drop=True).iterrows():
    plt.text(
        i, row["top_genre_prop"] + 0.02,
        str(row["top_genre"]),
        ha="center", va="bottom", fontsize=8, rotation=45
    )

plt.tight_layout()
plt.show()


## 16. Evaluate models with OvR ROC curves and macro AUC
Define a reusable evaluation function, then compare the logistic regression and calibrated SVM models on the test set.


In [ ]:
# =========================================================
# Utility: OvR AUC + ROC curve evaluation for multiclass models
# =========================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score, roc_curve

def evaluate_ovr_auc_and_roc(model, X_test, y_test, label_encoder, title="OvR ROC Curves"):
    """
    Computes per-class OvR ROC curves and AUCs on TEST data.
    Assumes model supports predict_proba (e.g. LogisticRegression,
    CalibratedClassifierCV).
    """
    # Encode labels using TRAIN-fitted encoder
    y_test_enc = label_encoder.transform(y_test)
    classes = label_encoder.classes_
    n_classes = len(classes)

    # Predicted probabilities
    y_score = model.predict_proba(X_test)

    # Binarize true labels
    y_test_bin = label_binarize(y_test_enc, classes=np.arange(n_classes))

    per_class_auc = {}
    plt.figure(figsize=(7, 6))

    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        auc = roc_auc_score(y_test_bin[:, i], y_score[:, i])
        per_class_auc[cls] = auc
        plt.plot(fpr, tpr, label=f"{cls} (AUC={auc:.3f})")

    # Macro AUC
    auc_macro = roc_auc_score(
        y_test_bin,
        y_score,
        multi_class="ovr",
        average="macro"
    )

    plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title}\nMacro OvR AUC = {auc_macro:.3f}")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

    return {
        "auc_macro": auc_macro,
        "per_class_auc": per_class_auc
    }


# =========================================================
# Step 10: Evaluate using OvR AUC + ROC curve
# (Reusable evaluation function for ANY multiclass classifier)
# =========================================================
baseline_metrics = evaluate_ovr_auc_and_roc(
    model=baseline_clf,
    X_test=X_test_final,
    y_test=y_test,
    label_encoder=le,
    title="Baseline Multinomial Logistic Regression (Final Features): OvR ROC Curves"
)

svm_metrics = evaluate_ovr_auc_and_roc(
    model=cal_svm,
    X_test=X_test_final,
    y_test=y_test,
    label_encoder=le,
    title="Calibrated Linear SVM (Final Features): OvR ROC Curves"
)


## 17. Error analysis
Use a confusion matrix and per-genre metrics to better understand where the final model performs well or struggles.


In [ ]:
# =========================================================
# Step 11: Error analysis (confusion matrix + per-genre AUC summary)
#   (LOGISTIC REGRESSION version)
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder, label_binarize

# -----------------------------
# 11A) Confusion matrix (TEST) — Logistic Regression
# -----------------------------
# Ensure consistent label encoding order (prefer existing le from training)
if "le" in globals():
    le_err = le
else:
    le_err = LabelEncoder().fit(y_train)

y_test_enc_err = le_err.transform(y_test)

# Predict labels using Logistic Regression
y_pred = baseline_clf.predict(X_test_final)  # baseline_clf was trained on encoded labels

# Convert to encoded ints if needed (usually already ints)
if y_pred.dtype.kind in ["U", "S", "O"]:
    y_pred_enc = le_err.transform(y_pred)
else:
    y_pred_enc = y_pred

cm = confusion_matrix(y_test_enc_err, y_pred_enc)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_norm,
    xticklabels=le_err.classes_,
    yticklabels=le_err.classes_,
    cmap="Blues",
    annot=False
)
plt.title("Normalized Confusion Matrix (TEST set) — Logistic Regression")
plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")
plt.tight_layout()
plt.show()

# -----------------------------
# 11B) Per-genre AUC table + bar chart — Logistic Regression
# -----------------------------
# Prefer reusing baseline_metrics from Step 10 if it exists
if "baseline_metrics" in globals() and "per_class_auc" in baseline_metrics:
    per_genre_auc = baseline_metrics["per_class_auc"]
else:
    # Compute from scratch using predict_proba from Logistic Regression
    y_test_proba = baseline_clf.predict_proba(X_test_final)
    classes = le_err.classes_
    n_classes = len(classes)

    y_test_bin = label_binarize(y_test_enc_err, classes=np.arange(n_classes))

    per_genre_auc = {}
    for i, g in enumerate(classes):
        per_genre_auc[g] = roc_auc_score(y_test_bin[:, i], y_test_proba[:, i])

auc_df = (
    pd.DataFrame.from_dict(per_genre_auc, orient="index", columns=["OvR_AUC"])
      .sort_values("OvR_AUC", ascending=False)
)

print("Per-genre OvR AUC (TEST set) — Logistic Regression:")
display(auc_df)

plt.figure(figsize=(8, 4))
auc_df["OvR_AUC"].plot(kind="bar")
plt.ylim(0.5, 1.0)
plt.ylabel("OvR AUC")
plt.title("Per-Genre AUC (TEST set) — Logistic Regression")
plt.tight_layout()
plt.show()

print("\nStep 11 complete: Logistic Regression error analysis finished.")


## 18. Final takeaways

### Final model
- **Selected model:** multinomial logistic regression
- **Reported macro OvR AUC:** **0.901**

### Why this pipeline worked
- invalid values were cleaned before modeling
- the train/test workflow avoided leakage
- numeric features were scaled appropriately
- PCA reduced dimensionality while keeping most signal
- KMeans added useful structural information without replacing the classification model

### Suggested GitHub repository files
For a strong GitHub upload, pair this notebook with:
- `README.md`
- `requirements.txt`
- optional figures exported from the notebook
- a short note about the dataset source and how to run the notebook

### Acknowledgment
AI was used to help debug and refine parts of the code.
